In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

sns.set_theme(style="whitegrid")

output_dir = 'visualization'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

missing_props = [10, 20, 30, 40, 50]
methods = ['KNN', 'MICE', 'Mean', 'MissForest', 'SHD']


nrmse_data = {
    'KNN': [0.6253, 0.8049, 0.7973, 1.0205, 0.9073],
    'MICE': [0.5383, 0.7181, 0.7308, 0.9646, 0.9077],
    'Mean': [0.8357, 0.9228, 0.8727, 1.0760, 0.9261],
    'MissForest': [0.4926, 0.6860, 0.7010, 0.8837, 0.8530],
    'SHD': [1.2494, 1.2711, 1.3189, 1.4145, 1.3832]
}


data = {
    'Logistic Regression': {
        'KNN': [0.7900, 0.7805, 0.7808, 0.7532, 0.7452],
        'MICE': [0.7893, 0.7868, 0.7809, 0.7717, 0.7746],
        'Mean': [0.7898, 0.7849, 0.7809, 0.7738, 0.7876],
        'MissForest': [0.7886, 0.7855, 0.7786, 0.7752, 0.7768],
        'SHD': [0.7802, 0.7594, 0.6690, 0.6509, 0.6611]
    },
    'Random Forest': {
        'KNN': [0.8434, 0.8442, 0.8435, 0.8390, 0.8324],
        'MICE': [0.8438, 0.8438, 0.8448, 0.8394, 0.8407],
        'Mean': [0.8458, 0.8458, 0.8433, 0.8396, 0.8374],
        'MissForest': [0.8412, 0.8437, 0.8368, 0.8355, 0.8314],
        'SHD': [0.8366, 0.8381, 0.8353, 0.8251, 0.8162]
    },
    'XGBoost': {
        'KNN': [0.8390, 0.8421, 0.8413, 0.8394, 0.8305],
        'MICE': [0.8406, 0.8428, 0.8427, 0.8427, 0.8448],
        'Mean': [0.8446, 0.8412, 0.8313, 0.8385, 0.8211],
        'MissForest': [0.8400, 0.8419, 0.8373, 0.8353, 0.8265],
        'SHD': [0.8340, 0.8326, 0.8339, 0.8199, 0.8080]
    }
}

fig, axes = plt.subplots(3, 2, figsize=(18, 18))
fig.suptitle('Performance of Imputation Methods (NRMSE & AUC)', fontsize=18, fontweight='bold', y=0.98)


colors = {'KNN': '#1f77b4', 'MICE': '#ff7f0e', 'Mean': '#2ca02c', 'MissForest': '#d62728', 'SHD': '#9467bd'}


def plot_line_chart(ax, model_name):
    for method in methods:
        ax.plot(missing_props, data[model_name][method], marker='o', linewidth=2, 
                label=method, color=colors[method])
    ax.set_title(f'{model_name} (AUC by Missing Proportion)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Missing Proportion (%)', fontsize=12)
    ax.set_ylabel('AUC Score', fontsize=12)
    ax.set_xticks(missing_props)
    ax.legend()

    ax.set_ylim(min([min(v) for v in data[model_name].values()]) - 0.01, 
                max([max(v) for v in data[model_name].values()]) + 0.01)
    

    fig_single, ax_single = plt.subplots(figsize=(8, 6))
    for method in methods:
        ax_single.plot(missing_props, data[model_name][method], marker='o', linewidth=2, 
                label=method, color=colors[method])
    ax_single.set_title(f'{model_name} (AUC by Missing Proportion)', fontsize=14, fontweight='bold')
    ax_single.set_xlabel('Missing Proportion (%)', fontsize=12)
    ax_single.set_ylabel('AUC Score', fontsize=12)
    ax_single.set_xticks(missing_props)
    ax_single.legend()
    ax_single.set_ylim(min([min(v) for v in data[model_name].values()]) - 0.01, 
                max([max(v) for v in data[model_name].values()]) + 0.01)
    fig_single.savefig(f'{output_dir}/{model_name.replace(" ", "_")}_AUC.png', bbox_inches='tight')
    plt.close(fig_single)

# ---------------------------------------------------------
# พล็อตกราฟ NRMSE (Approach 1)
ax_nrmse = axes[0, 0]
def plot_nrmse_chart(ax_target):
    for method in methods:
        ax_target.plot(missing_props, nrmse_data[method], marker='D', markersize=7, linewidth=2, 
                label=method, color=colors[method])
    ax_target.set_title('NRMSE of Imputation Methods (Lower is Better)', fontsize=14, fontweight='bold')
    ax_target.set_xlabel('Missing Proportion (%)', fontsize=12)
    ax_target.set_ylabel('NRMSE Score', fontsize=12)
    ax_target.set_xticks(missing_props)
    ax_target.legend(title='Imputation Method')

plot_nrmse_chart(ax_nrmse)


fig_nrmse, ax_nrmse_single = plt.subplots(figsize=(8, 6))
plot_nrmse_chart(ax_nrmse_single)
fig_nrmse.savefig(f'{output_dir}/NRMSE_Performance.png', bbox_inches='tight')
plt.close(fig_nrmse)


plot_line_chart(axes[0, 1], 'Logistic Regression')

plot_line_chart(axes[1, 0], 'Random Forest')

plot_line_chart(axes[1, 1], 'XGBoost')


avg_over_classifiers = {method: [] for method in methods}
for method in methods:
    for i in range(len(missing_props)):

        avg_val = np.mean([data[model][method][i] for model in data.keys()])
        avg_over_classifiers[method].append(avg_val)

ax_line_avg = axes[2, 0]

axes[2, 1].axis('off')

def plot_average_line_chart(ax_target):
    for method in methods:
        ax_target.plot(missing_props, avg_over_classifiers[method], marker='s', markersize=8, linewidth=2.5, 
                label=method, color=colors[method])

    ax_target.set_title('Average AUC over Three Classifiers', fontsize=14, fontweight='bold')
    ax_target.set_xlabel('Missing Proportion (%)', fontsize=12)
    ax_target.set_ylabel('Average AUC Score', fontsize=12)
    ax_target.set_xticks(missing_props)
    ax_target.legend(title='Imputation Method')
    

    all_avg_vals = [val for vals in avg_over_classifiers.values() for val in vals]
    min_avg = min(all_avg_vals)
    max_avg = max(all_avg_vals)
    padding = (max_avg - min_avg) * 0.15
    if padding == 0: padding = 0.05
    ax_target.set_ylim(min_avg - padding, max_avg + padding)

plot_average_line_chart(ax_line_avg)

fig_avg, ax_avg_single = plt.subplots(figsize=(8, 6))
plot_average_line_chart(ax_avg_single)
fig_avg.savefig(f'{output_dir}/Average_AUC_over_Three_Classifiers.png', bbox_inches='tight')
plt.close(fig_avg)

plt.tight_layout()
plt.subplots_adjust(top=0.92)

fig.savefig(f'{output_dir}/All_Charts_Combined.png', bbox_inches='tight')

plt.show()
print(f"กราฟทั้งหมดถูกบันทึกไว้ที่โฟลเดอร์: {output_dir}")